# Transformers


#### A Transformer is a deep learning architecture designed to process sequences of text using self-attention.

- Transformers are the foundation of many modern NLP models, including BERT, GPT, and T5.

- Before Transformers, we used models such as RNNs and LSTMs for NLP tasks.

<img src="https://drive.google.com/thumbnail?id=1unPqGaKftHxp37-F99Bg6hJzFbQdCGC7&sz=w1000" width="700">

In [1]:
%pip -q install transformers torch matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load a Transformer

We use **DistilBERT**, a  Transformer fine-tuned for positive/negative sentiment classification.

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

# Look inside the Transformer architecture.
print("Transformer layers:", model.config.n_layers)
print("Attention heads per layer:", model.config.n_heads)
print("Hidden size:", model.config.dim)

C:\Users\fedaa\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 457.77it/s]

Transformer layers: 6
Attention heads per layer: 12
Hidden size: 768



<img src="https://drive.google.com/thumbnail?id=1v2dRK1KR_ydZOknjj8Xv8_wjOdXleDYs&sz=w1000" width="500">

## 2. Tokenizing the text

In [3]:
sentence = "The movie was not good."

# Tokenization: converts text into token IDs
inputs = tokenizer(sentence, return_tensors="pt")

# [CLS] = start token, [SEP] = end token
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:", tokens)

Tokens: ['[CLS]', 'the', 'movie', 'was', 'not', 'good', '.', '[SEP]']


## 3. Embeddings and self-attention

Each token ID becomes a **vector (embedding)**. Inside each Transformer layer, **self-attention** lets a token use information from other tokens in the same sentence.

In [4]:
import torch

# Enable attention weights
model.set_attn_implementation("eager")

with torch.no_grad():
    embeddings = model.distilbert.embeddings(inputs["input_ids"])
    outputs = model(**inputs, output_attentions=True)

# Attention from the first layer and first head
attention = outputs.attentions[0][0, 0]

print("Embeddings shape:", embeddings.shape)
print("Attention shape:", attention.shape)
print("Tokens:", tokens)
print("Attention weights:\n", attention)

Embeddings shape: torch.Size([1, 8, 768])
Attention shape: torch.Size([8, 8])
Tokens: ['[CLS]', 'the', 'movie', 'was', 'not', 'good', '.', '[SEP]']
Attention weights:
 tensor([[0.1145, 0.0938, 0.0681, 0.0701, 0.0596, 0.0800, 0.1593, 0.3546],
        [0.4147, 0.0793, 0.0481, 0.1018, 0.1084, 0.0805, 0.0789, 0.0883],
        [0.1634, 0.0327, 0.0975, 0.0403, 0.1819, 0.1402, 0.0866, 0.2574],
        [0.2882, 0.1014, 0.0504, 0.0669, 0.2180, 0.1009, 0.1231, 0.0511],
        [0.3612, 0.0501, 0.0696, 0.1200, 0.0642, 0.1304, 0.0953, 0.1092],
        [0.3065, 0.0469, 0.1175, 0.1915, 0.0630, 0.0555, 0.1120, 0.1071],
        [0.3505, 0.0935, 0.0572, 0.0753, 0.0972, 0.0964, 0.1165, 0.1135],
        [0.3513, 0.1019, 0.0318, 0.0541, 0.1036, 0.0607, 0.0755, 0.2210]])


## 4. Classification: From Transformer to prediction

The Transformer produces contextual representations. The **classification head** turns them into two scores (*logits*). **Softmax** converts these into probabilities that sum to 1.

In [5]:
# Get prediction scores from the Transformer
with torch.no_grad():
    logits = model(**inputs).logits

# Convert scores into probabilities
probs = torch.softmax(logits, dim=1)[0]

# Display the prediction
prediction = probs.argmax().item()

print("Prediction:", model.config.id2label[prediction])
print("Confidence:", round(probs[prediction].item(), 3))


Prediction: NEGATIVE
Confidence: 1.0



### Task: Exploring Transformer Layers and Attention Heads

Load the pretrained **`bert-base-uncased`** model using Hugging Face Transformers.

1. Write code to print the number of Transformer layers and the number of attention heads per layer.
2. Calculate and print the total number of attention heads across all Transformer layers.
3. Using the sentence `"Transformers use self-attention to understand language."`, extract the attention weights from the **third Transformer layer and second attention head**.
4. Explain what the rows and columns of the attention matrix represent.

In [6]:

import torch
from transformers import AutoModel, AutoTokenizer

# Load pretrained BERT model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, attn_implementation="eager")
model.eval()

# 1. Number of layers and attention heads
num_layers = model.config.num_hidden_layers
num_heads = model.config.num_attention_heads

print("Number of Transformer layers:", num_layers)
print("Number of attention heads per layer:", num_heads)

# 2. Total number of attention heads
total_heads = num_layers * num_heads
print("Total number of attention heads:", total_heads)

# 3. Extract attention weights
sentence = "Transformers use self-attention to understand language."
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attention_matrix = outputs.attentions[2][0, 1]

print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Attention matrix shape:", attention_matrix.shape)
print("Attention weights:")
print(attention_matrix)

# 4. Explain rows and columns
print("Rows represent query tokens.")
print("Columns represent key tokens.")
print("Each value represents how much attention one token gives to another.")


C:\Users\fedaa\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fedaa\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\fedaa\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:149: UserWarn

Number of Transformer layers: 12
Number of attention heads per layer: 12
Total number of attention heads: 144
Tokens: ['[CLS]', 'transformers', 'use', 'self', '-', 'attention', 'to', 'understand', 'language', '.', '[SEP]']
Attention matrix shape: torch.Size([11, 11])
Attention weights:
tensor([[7.3176e-01, 7.9132e-03, 1.0591e-02, 5.5538e-03, 1.6532e-02, 6.8604e-03,
         1.0085e-02, 1.1588e-02, 5.5202e-03, 3.8613e-02, 1.5499e-01],
        [8.8883e-01, 4.7016e-03, 3.4382e-02, 1.7840e-04, 2.7377e-03, 5.0542e-03,
         9.0185e-04, 7.3192e-04, 4.0114e-04, 5.1106e-03, 5.6975e-02],
        [1.4273e-01, 5.6892e-01, 5.4059e-03, 3.5340e-02, 4.1156e-02, 6.7886e-02,
         8.9602e-02, 3.2741e-04, 5.3299e-03, 8.9671e-03, 3.4334e-02],
        [3.3953e-01, 7.3164e-02, 1.5250e-03, 1.6106e-04, 2.0160e-01, 8.1124e-02,
         6.3570e-02, 2.5709e-02, 1.6392e-03, 2.8494e-02, 1.8348e-01],
        [1.4291e-01, 6.9677e-02, 7.1289e-02, 6.3943e-01, 1.2104e-02, 1.2819e-02,
         3.4381e-03, 2.5649e

In [7]:

import pandas as pd
from IPython.display import display

results = pd.DataFrame({
    "Question": [
        "Q1",
        "Q1",
        "Q2",
        "Q3",
        "Q3",
        "Q4",
        "Q4"
    ],
    "Description": [
        "Number of Transformer layers",
        "Attention heads per layer",
        "Total attention heads",
        "Selected Transformer layer",
        "Selected attention head",
        "Rows represent",
        "Columns represent"
    ],
    "Result": [
        num_layers,
        num_heads,
        total_heads,
        "Third layer (Index 2)",
        "Second head (Index 1)",
        "Query tokens",
        "Key tokens"
    ]
})

display(results)


,Question,Description,Result
0,Q1,Number of Transformer layers,12
1,Q1,Attention heads per layer,12
2,Q2,Total attention heads,144
3,Q3,Selected Transformer layer,Third layer (Index 2)
4,Q3,Selected attention head,Second head (Index 1)
5,Q4,Rows represent,Query tokens
6,Q4,Columns represent,Key tokens


In [8]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

attention_df = pd.DataFrame(
    attention_matrix.detach().cpu().numpy(),
    index=tokens,
    columns=tokens
)

display(attention_df.style.format("{:.4f}"))

,[CLS],transformers,use,self,-,attention,to,understand,language,.,[SEP]
[CLS],0.7318,0.0079,0.0106,0.0056,0.0165,0.0069,0.0101,0.0116,0.0055,0.0386,0.1550
transformers,0.8888,0.0047,0.0344,0.0002,0.0027,0.0051,0.0009,0.0007,0.0004,0.0051,0.0570
use,0.1427,0.5689,0.0054,0.0353,0.0412,0.0679,0.0896,0.0003,0.0053,0.0090,0.0343
self,0.3395,0.0732,0.0015,0.0002,0.2016,0.0811,0.0636,0.0257,0.0016,0.0285,0.1835
-,0.1429,0.0697,0.0713,0.6394,0.0121,0.0128,0.0034,0.0026,0.0142,0.0093,0.0223
attention,0.1126,0.5448,0.0116,0.0203,0.1112,0.0102,0.0724,0.0001,0.0001,0.0164,0.1003
to,0.0841,0.0224,0.6222,0.0085,0.0091,0.1936,0.0110,0.0072,0.0008,0.0140,0.0271
understand,0.4336,0.0050,0.0023,0.0216,0.0119,0.0030,0.3296,0.0024,0.0554,0.0533,0.0820
language,0.0513,0.0005,0.4292,0.0016,0.0045,0.0013,0.0051,0.3980,0.0059,0.0407,0.0620
.,0.6580,0.0056,0.0031,0.0011,0.0044,0.0025,0.0094,0.0379,0.0459,0.0726,0.1594
